# So Sánh 3 Model Phân Loại Ảnh — EfficientNet / ResNet / ViT

**Task:** Kiểm duyệt ảnh đa loại — 3 class:
- `SAFE` (0) — ảnh bình thường
- `NSFW` (1) — nội dung 18+, khiêu dâm
- `VIOLENCE` (2) — bạo lực, máu me

**Dataset (2 nguồn, tổng ~30,000 ảnh):**
- `Falconsai/nsfw_image_detection` → class SAFE + NSFW (~25k)
- `Real Life Violence Situations` (Kaggle) → class VIOLENCE (~5k)

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 x2** hoặc P100
- Internet: **ON**
- Add dataset: tìm **"Real Life Violence Situations Dataset"** trong Kaggle Datasets và thêm vào notebook

**Ước tính:** ~1–2h/model × 3 model = ~4–6h tổng

In [ ]:
!pip install timm datasets scikit-learn -q

In [ ]:
import os, json, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ════════════════════════════════════════════════════════════
# CẤU HÌNH
# ════════════════════════════════════════════════════════════
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 64   # mỗi GPU
EPOCHS      = 10
LR          = 3e-4
NUM_WORKERS = 2
PATIENCE    = 3    # early stopping

# 3 model cần so sánh
MODELS = {
    'efficientnet_b0' : 'EfficientNet-B0',
    'resnet50'        : 'ResNet-50',
    'vit_base_patch16_224': 'ViT-Base',
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ════════════════════════════════════════════════════════════
# LOAD & GỘP DATASET — 3 CLASS: SAFE / NSFW / VIOLENCE
# ════════════════════════════════════════════════════════════

CLASS_NAMES = ['SAFE', 'NSFW', 'VIOLENCE']
NUM_CLASSES = 3
LABEL_SAFE     = 0
LABEL_NSFW     = 1
LABEL_VIOLENCE = 2

# Giới hạn số ảnh mỗi class để cân bằng (~10k/class = 30k tổng)
MAX_PER_CLASS = 10_000

all_images = []   # list of (PIL.Image, int label)

# ── Nguồn 1: Falconsai NSFW ──────────────────────────────────────────────────
print('Loading Falconsai/nsfw_image_detection...')
nsfw_ds = load_dataset('Falconsai/nsfw_image_detection', split='train')

# Xem class gốc của dataset này
orig_names = nsfw_ds.features['label'].names
print(f'  Classes gốc: {orig_names}')

# Map class gốc → 3 class của chúng ta
# Falconsai thường có: neutral, drawings, hentai, porn, sexy
NSFW_MAP = {}
for i, name in enumerate(orig_names):
    n = name.lower()
    if 'neutral' in n or 'normal' in n or 'safe' in n:
        NSFW_MAP[i] = LABEL_SAFE
    elif any(x in n for x in ['porn', 'hentai', 'sexy', 'nsfw', '18', 'adult', 'explicit']):
        NSFW_MAP[i] = LABEL_NSFW
    elif 'draw' in n or 'cartoon' in n or 'anime' in n:
        NSFW_MAP[i] = LABEL_NSFW   # drawings có nội dung 18+ → NSFW
    else:
        NSFW_MAP[i] = LABEL_SAFE   # không xác định → SAFE

print(f'  Label mapping: {dict(zip(orig_names, NSFW_MAP.values()))}')

safe_count = nsfw_count = 0
for item in tqdm(nsfw_ds, desc='  Processing NSFW dataset'):
    mapped = NSFW_MAP[item['label']]
    if mapped == LABEL_SAFE and safe_count < MAX_PER_CLASS:
        all_images.append((item['image'], LABEL_SAFE))
        safe_count += 1
    elif mapped == LABEL_NSFW and nsfw_count < MAX_PER_CLASS:
        all_images.append((item['image'], LABEL_NSFW))
        nsfw_count += 1

print(f'  → SAFE: {safe_count:,} | NSFW: {nsfw_count:,}')

# ── Nguồn 2: Violence Dataset (Kaggle) ───────────────────────────────────────
# Dataset: "Real Life Violence Situations Dataset"
# Sau khi Add vào notebook, đường dẫn thường là:
VIOLENCE_DIRS = [
    '/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset/Violence',
    '/kaggle/input/real-life-violence-situations-dataset/Violence',
    '/kaggle/input/violence-image-dataset/violence',
    '/kaggle/input/violence-detection-dataset/violence',
]

violence_dir = None
for d in VIOLENCE_DIRS:
    if os.path.isdir(d):
        violence_dir = d
        break

if violence_dir:
    print(f'\nLoading violence images từ: {violence_dir}')
    violence_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    violence_files = [
        f for f in Path(violence_dir).rglob('*')
        if f.suffix.lower() in violence_exts
    ]
    random.shuffle(violence_files)
    violence_count = 0
    for fpath in tqdm(violence_files[:MAX_PER_CLASS], desc='  Processing Violence'):
        try:
            img = Image.open(fpath).convert('RGB')
            all_images.append((img, LABEL_VIOLENCE))
            violence_count += 1
        except Exception:
            continue
    print(f'  → VIOLENCE: {violence_count:,}')
else:
    print('\n[!] Không tìm thấy violence dataset!')
    print('    Vào Add Data > tìm "Real Life Violence Situations Dataset" > Add')
    print('    Sau đó chạy lại cell này.')
    print('\n    Tạm dùng placeholder 100 ảnh SAFE làm VIOLENCE để test pipeline...')
    # Fallback tạm thời để test pipeline không bị lỗi
    for item in nsfw_ds.select(range(100)):
        all_images.append((item['image'], LABEL_VIOLENCE))

# ── Thống kê cuối ────────────────────────────────────────────────────────────
from collections import Counter
label_dist = Counter(lbl for _, lbl in all_images)
print(f'\nTổng dataset:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  [{i}] {name:10s}: {label_dist[i]:,}')
print(f'  TỔNG      : {len(all_images):,}')

# Shuffle
random.shuffle(all_images)

In [ ]:
# ════════════════════════════════════════════════════════════
# DATASET CLASS + TRANSFORMS + DATALOADER
# ════════════════════════════════════════════════════════════
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class ImageDataset(Dataset):
    def __init__(self, data, transform=None):
        # data: list of (PIL.Image, int label)
        self.data      = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image, label = self.data[idx]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Train / Val / Test split = 80 / 10 / 10
total   = len(all_images)
n_val   = int(total * 0.1)
n_test  = int(total * 0.1)
n_train = total - n_val - n_test

train_data = all_images[:n_train]
val_data   = all_images[n_train:n_train + n_val]
test_data  = all_images[n_train + n_val:]

train_ds = ImageDataset(train_data, train_transform)
val_ds   = ImageDataset(val_data,   val_transform)
test_ds  = ImageDataset(test_data,  val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')

# Kiểm tra phân phối trong tập train
train_dist = Counter(lbl for _, lbl in train_data)
print('Phân phối train:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name}: {train_dist[i]:,}')

In [ ]:
# ════════════════════════════════════════════════════════════
# HÀM TRAIN CHUNG (dùng cho cả 3 model)
# ════════════════════════════════════════════════════════════

def train_one_model(model_key, model_name):
    print(f'\n{"="*60}')
    print(f'Training: {model_name}  ({model_key})')
    print(f'{"="*60}')

    # ── Load pretrained model từ timm ──────────────────────
    model = timm.create_model(
        model_key,
        pretrained=True,
        num_classes=NUM_CLASSES
    )

    # Dùng DataParallel nếu có nhiều GPU
    if torch.cuda.device_count() > 1:
        print(f'  DataParallel: {torch.cuda.device_count()} GPUs')
        model = nn.DataParallel(model)
    model = model.to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Parameters: {n_params:.1f}M')

    # ── Optimizer & Scheduler ──────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    # ── Training Loop ──────────────────────────────────────
    best_val_acc = 0.0
    patience_cnt = 0
    history      = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_path    = OUTPUT_DIR / f'{model_key}_best.pt'

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        # Train
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                logits = model(imgs)
                val_loss += criterion(logits, labels).item()
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader)
        val_acc   = accuracy_score(all_labels, all_preds) * 100
        val_f1    = f1_score(all_labels, all_preds, average='weighted') * 100

        scheduler.step()

        elapsed = time.time() - t0
        print(f'  Epoch {epoch:2d}/{EPOCHS} | '
              f'loss={train_loss:.4f} | '
              f'val_loss={val_loss:.4f} | '
              f'val_acc={val_acc:.2f}% | '
              f'val_f1={val_f1:.2f}% | '
              f'{elapsed:.0f}s')

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Early stopping + lưu best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_cnt = 0
            raw_model = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(raw_model.state_dict(), best_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f'  Early stopping tại epoch {epoch}')
                break

    # ── Đánh giá trên Test set ─────────────────────────────
    print(f'\n  Load best model (val_acc={best_val_acc:.2f}%)...')
    raw_model = timm.create_model(model_key, pretrained=False, num_classes=NUM_CLASSES)
    raw_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    raw_model = raw_model.to(DEVICE)
    raw_model.eval()

    test_preds, test_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            test_preds.extend(raw_model(imgs).argmax(1).cpu().numpy())
            test_labels.extend(labels.numpy())

    test_acc = accuracy_score(test_labels, test_preds) * 100
    test_f1  = f1_score(test_labels, test_preds, average='weighted') * 100
    print(f'  TEST  acc={test_acc:.2f}%  f1={test_f1:.2f}%')
    print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

    return {
        'model_name' : model_name,
        'model_key'  : model_key,
        'best_val_acc': best_val_acc,
        'test_acc'   : test_acc,
        'test_f1'    : test_f1,
        'history'    : history,
        'test_preds' : test_preds,
        'test_labels': test_labels,
        'best_path'  : str(best_path),
    }

print('Hàm train đã sẵn sàng.')

In [ ]:
# ════════════════════════════════════════════════════════════
# TRAIN CẢ 3 MODEL
# ════════════════════════════════════════════════════════════
all_results = {}

for model_key, model_name in MODELS.items():
    result = train_one_model(model_key, model_name)
    all_results[model_key] = result
    # Giải phóng VRAM trước khi train model tiếp
    torch.cuda.empty_cache()

print('\n✓ Đã train xong cả 3 model.')

In [ ]:
# ════════════════════════════════════════════════════════════
# BẢNG SO SÁNH KẾT QUẢ
# ════════════════════════════════════════════════════════════
print('\n' + '='*55)
print(f'{"Model":<22} {"Val Acc":>10} {"Test Acc":>10} {"Test F1":>10}')
print('='*55)
for key, r in all_results.items():
    print(f'{r["model_name"]:<22} {r["best_val_acc"]:>9.2f}% {r["test_acc"]:>9.2f}% {r["test_f1"]:>9.2f}%')
print('='*55)

best_key = max(all_results, key=lambda k: all_results[k]['test_acc'])
print(f'\nModel tốt nhất: {all_results[best_key]["model_name"]} '
      f'(test_acc={all_results[best_key]["test_acc"]:.2f}%)')

# Lưu kết quả ra JSON
summary = {k: {kk: vv for kk, vv in v.items() if kk != 'history'}
           for k, v in all_results.items()}
with open(OUTPUT_DIR / 'comparison_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Đã lưu comparison_results.json')

In [ ]:
# ════════════════════════════════════════════════════════════
# BIỂU ĐỒ SO SÁNH
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2196F3', '#4CAF50', '#FF5722']

for ax, (key, r), color in zip(axes, all_results.items(), colors):
    epochs_ran = range(1, len(r['history']['val_acc']) + 1)
    ax.plot(epochs_ran, r['history']['train_loss'], label='Train Loss', linestyle='--', color=color, alpha=0.6)
    ax2 = ax.twinx()
    ax2.plot(epochs_ran, r['history']['val_acc'], label='Val Acc', color=color, linewidth=2)
    ax.set_title(f"{r['model_name']}\nTest Acc: {r['test_acc']:.2f}%", fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='gray')
    ax2.set_ylabel('Val Accuracy (%)', color=color)
    ax2.set_ylim(0, 100)

plt.suptitle('So sánh 3 Model — Training curve', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# CONFUSION MATRIX CHO CẢ 3 MODEL
# ════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (key, r), color in zip(axes, all_results.items(), colors):
    cm = confusion_matrix(r['test_labels'], r['test_preds'])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', ax=ax,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', cbar=False
    )
    ax.set_title(f"{r['model_name']}\nTest Acc: {r['test_acc']:.2f}%")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrix (%) — So sánh 3 Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

In [ ]:
# ════════════════════════════════════════════════════════════
# LƯU MODEL TỐT NHẤT để dùng trong production
# ════════════════════════════════════════════════════════════
import shutil

best_key  = max(all_results, key=lambda k: all_results[k]['test_acc'])
best_info = all_results[best_key]
print(f'Model tốt nhất: {best_info["model_name"]} — test_acc={best_info["test_acc"]:.2f}%')

# Lưu model tốt nhất + metadata vào thư mục riêng
best_dir = OUTPUT_DIR / 'best_image_model'
best_dir.mkdir(exist_ok=True)

shutil.copy(best_info['best_path'], best_dir / 'model.pt')

meta = {
    'model_key'  : best_key,
    'model_name' : best_info['model_name'],
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'img_size'   : IMG_SIZE,
    'test_acc'   : best_info['test_acc'],
    'test_f1'    : best_info['test_f1'],
    'mean'       : MEAN,
    'std'        : STD,
}
with open(best_dir / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

# Zip để download
shutil.make_archive(str(OUTPUT_DIR / 'best_image_model'), 'zip', str(best_dir))

import os
zip_size = os.path.getsize(str(OUTPUT_DIR / 'best_image_model.zip')) / 1024**2
print(f'Saved: best_image_model.zip ({zip_size:.0f} MB)')
print('\nVào tab Output của Kaggle để download.')

In [ ]:
# ════════════════════════════════════════════════════════════
# TEST THỬ INFERENCE với 1 ảnh
# ════════════════════════════════════════════════════════════
import urllib.request

# Load model tốt nhất
infer_model = timm.create_model(best_key, pretrained=False, num_classes=NUM_CLASSES)
infer_model.load_state_dict(torch.load(best_dir / 'model.pt', map_location=DEVICE))
infer_model = infer_model.to(DEVICE)
infer_model.eval()

def predict_image(image_path_or_pil):
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        img = image_path_or_pil.convert('RGB')

    tensor = val_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = infer_model(tensor)
        probs  = torch.softmax(logits, dim=1)[0].cpu().numpy()

    pred_idx = probs.argmax()
    return {
        'label'      : CLASS_NAMES[pred_idx],
        'confidence' : float(probs[pred_idx]),
        'all_scores' : {CLASS_NAMES[i]: float(p) for i, p in enumerate(probs)},
    }

# Test với 3 ảnh ngẫu nhiên từ test set
for i in random.sample(range(len(test_split)), 3):
    sample_img = test_split[i]['image']
    true_label = CLASS_NAMES[test_split[i]['label']]
    result     = predict_image(sample_img)
    print(f'Thực tế: {true_label:10s} | Dự đoán: {result["label"]:10s} | Confidence: {result["confidence"]*100:.1f}%')